In [ ]:
!pip install -q openpyxl yfinance mplfinance
!pip install -q fpdf
!pip install --upgrade yfinance
from fpdf import FPDF
import tempfile
import requests
from PIL import Image
from io import BytesIO
import seaborn as sns
import matplotlib.pyplot as plt
import yfinance as yf
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import matplotlib
matplotlib.style.use('dark_background')
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# --- Dictionnaire ---
secteurs_actions = {
    'Technologie': ['AAPL', 'MSFT', 'NVDA'],
    'Banque': ['JPM', 'BAC', 'GS'],
    'Santé': ['JNJ', 'PFE', 'MRK'],
    'Immobilier': ['AMT', 'PLD', 'CCI'],
    'Consommation': ['PG', 'KO', 'PEP'],
    'Automobile': ['TSLA', 'F', 'GM'],
    'Énergie': ['XOM', 'CVX', 'COP']
}

secteurs_indices = {
    'Technologie': '^IXIC',
    'Banque': '^BKX',
    'Santé': 'XLV',
    'Immobilier': 'XLRE',
    'Consommation': 'XLP',
    'Automobile': 'CARZ',
    'Énergie': 'XLE'
}

# --- Scroll secteur ---
secteur_dropdown = widgets.Dropdown(
    options=list(secteurs_actions.keys()),
    description='Secteur :',
    value=None,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# --- Scroll profil ---
profil_dropdown = widgets.Dropdown(
    options=['Prudent', 'Équilibré', 'Dynamique'],
    description='Profil :',
    value=None,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# --- Insrtion montant ---
montant_input = widgets.FloatText(
    description="Montant à investir ($):",
    value=1000.0,
    step=100.0,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# --- Calculs financiers ---
def calculs_financiers(sector):
    tickers = secteurs_actions[sector]
    indice = secteurs_indices[sector]
    all_tickers = tickers + [indice]

    data = yf.download(all_tickers, period="1y", interval='1d', progress=False)['Close']
    data_monthly = data.resample('ME').last()
    returns = np.log(data_monthly / data_monthly.shift(1)).dropna()
    actions_returns = returns[tickers]
    indice_returns = returns[indice]

    betas = {ticker: [] for ticker in tickers}
    for i in range(1, len(data_monthly)):
        for ticker in tickers:
            r_stock = actions_returns[ticker].iloc[:i+1]
            r_index = indice_returns.iloc[:i+1]
            beta = np.cov(r_stock, r_index)[0][1] / np.var(r_index)
            betas[ticker].append(beta)
    beta_df = pd.DataFrame(betas, index=data_monthly.index[1:])

    corr_matrix = actions_returns.corr()
    ticker_names = {t: yf.Ticker(t).info.get('longName', t) for t in tickers}
    corr_matrix_named = corr_matrix.rename(index=ticker_names, columns=ticker_names)

    betas_df = pd.DataFrame(betas, index=data_monthly.index[1:]).rename(columns=ticker_names)
    base100_df = ((data / data.iloc[0]) * 100)[tickers].rename(columns=ticker_names)

    fiche_data = []
    for ticker in tickers:
        info = yf.Ticker(ticker).info
        fiche_data.append({
            'Nom de l’entreprise': info.get('longName', ticker),
            'Ticker': ticker,
            'Industrie': info.get('industry', 'N/A'),
            'Capitalisation (M$)': round(info.get('marketCap', 0) / 1e6, 2),
            'Chiffre d’affaires (M$)': round(info.get('totalRevenue', 0) / 1e6, 2),
            'Bénéfice net (M$)': round(info.get('netIncomeToCommon', 0) / 1e6, 2),
            'PER': round(info.get('trailingPE', 0), 2),
            'Rendement du dividende (%)': round(info.get('dividendYield', 0)*100, 2) if info.get('dividendYield') else "N/A",
        })

    fiche_df = pd.DataFrame(fiche_data)

    with pd.ExcelWriter(f"Analyse financière en {sector}.xlsx", engine='openpyxl') as writer:
        base100_df.to_excel(writer, sheet_name="Base 100")
        betas_df.to_excel(writer, sheet_name="Bêtas Mensuels")
        corr_matrix_named.to_excel(writer, sheet_name="Corrélation")
        fiche_df.to_excel(writer, sheet_name="Fiche entreprises", index=False)

    wb = load_workbook(f"Analyse financière en {sector}.xlsx")
    thin_border = Border(left=Side(style='thin'), right=Side(style='thin'),
                         top=Side(style='thin'), bottom=Side(style='thin'))

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for row in ws.iter_rows():
            for cell in row:
                cell.alignment = Alignment(horizontal='center', vertical='center')
                cell.border = thin_border
                if isinstance(cell.value, float):
                    cell.number_format = '0.00'

        for cell in ws[1]:
            cell.fill = PatternFill(start_color="00336699", end_color="00336699", fill_type="solid")
            cell.font = Font(bold=True, color="FFFFFF")

        for column_cells in ws.columns:
            max_length = max(len(str(cell.value)) if cell.value else 0 for cell in column_cells)
            col_letter = column_cells[0].column_letter
            ws.column_dimensions[col_letter].width = max(10, max_length + 2)

    wb.save(f"Analyse financière en {sector}.xlsx")
    return data[tickers], base100_df, corr_matrix_named, fiche_df

#---Modèle De Markowitz
def optimiser_portefeuille(returns, profil, montant):
    cov_matrix = returns.cov()
    mean_returns = returns.mean()
    num_assets = len(mean_returns)
    weights = np.ones(num_assets) / num_assets

    if profil == 'Prudent':
        target_vol = 0.15
    elif profil == 'Équilibré':
        target_vol = 0.25
    else:
        target_vol = 0.40

    def port_perf(w):
        return w @ mean_returns, np.sqrt(w.T @ cov_matrix @ w)

    def objective(w):
        _, vol = port_perf(w)
        return (vol - target_vol) ** 2

    from scipy.optimize import minimize
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = [(0, 1)] * num_assets
    result = minimize(objective, weights, bounds=bounds, constraints=constraints)
    optimal_weights = result.x

    allocation = pd.Series(optimal_weights, index=mean_returns.index)
    allocation_dollars = allocation * montant

    return allocation.round(3), allocation_dollars.round(2)


# --- Graphiques
def plot_sector_charts(sector):
    tickers = secteurs_actions[sector]
    ticker_names = {t: yf.Ticker(t).info.get('longName', t) for t in tickers}
    data = yf.download(tickers, period="1y", interval='1d', progress=False)['Close']
    if data.empty:
        print("Aucune donnée trouvée.")
        return

    plt.figure(figsize=(14, 6))
    for t in tickers:
        plt.plot(data[t], label=ticker_names[t], linewidth=2)
    plt.title(f"Cours des actions - Secteur {sector}")
    plt.xlabel("Date")
    plt.ylabel("Prix ($)")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    base100 = (data / data.iloc[0]) * 100
    plt.figure(figsize=(14, 6))
    for t in tickers:
        plt.plot(base100[t], label=ticker_names[t], linewidth=2)
    plt.title(f"Performance (Base 100) - Secteur {sector}")
    plt.xlabel("Date")
    plt.ylabel("Index (Base 100)")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def generer_conclusion_ia(sector, fiche_df, base100, corr_matrix_named, profil, montant, actions_returns):
    entreprise = fiche_df.iloc[0]
    meilleure_perf = base100.iloc[-1].idxmax()
    perf_max = base100.iloc[-1].max()
    corr_moyenne = np.mean(corr_matrix_named.values[np.triu_indices_from(corr_matrix_named.values, k=1)])

    conseil = {
        'Prudent': "Nous vous recommandons d'investir dans des entreprises stables et bien établies afin d'éviter le plus de risque possible.",
        'Équilibré': "Un portefeuille diversifié dans ce secteur peut offrir un bon compromis entre risque et rendement tout ce que vous cherchez quoi :) .",
        'Dynamique': "Les entreprises les plus performantes de ce secteur pourraient offrir un fort potentiel de croissance ce qui conviedrait parafaitement avec votre mentalité."
    }

    rendement_decimal = (perf_max - 100) / 100
    profit_estime = montant * rendement_decimal

    #Opti de portefeuille
    allocation_pct, allocation_dollars = optimiser_portefeuille(actions_returns, profil, montant)

    recommandation = "\n\nOn vous propose cette répartition optimale de votre portefeuille :\n"
    for ticker in allocation_pct.index:
        recommandation += f"- {ticker}: {allocation_pct[ticker]*100:.1f}% soit {allocation_dollars[ticker]:.2f} $\n"

    return (
        f"Sur l'année écoulée, le secteur {sector} montre une performance variée.\n"
        f"L'entreprise la plus performante est {meilleure_perf} avec +{perf_max - 100:.1f}%.\n"
        f"La corrélation moyenne entre les actions est de {corr_moyenne:.2f}, ce qui indique "
        f"{'une forte interdépendance' if corr_moyenne > 0.6 else 'une bonne diversification'}.\n"
        f"{entreprise['Nom de l’entreprise']} se distingue avec une capitalisation de {entreprise['Capitalisation (M$)']:.0f} M$.\n"
        f"{conseil[profil]}\n\n"
        f"Pour un investissement de {montant:.2f} $, le rendement estimé est de {rendement_decimal * 100:.1f}% soit un profit de {profit_estime:.2f} $."
        f"{recommandation}"
    )

class PDF(FPDF):
    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(100, 100, 100)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

def export_pdf(sector, data, base100, corr_matrix_named, fiche_df, profil, montant):
    def safe_text(text):
        return str(text).encode('latin-1', 'replace').decode('latin-1')

    tmpdir = tempfile.mkdtemp()

    # === Graphiques
    graph1_path = f"{tmpdir}/cours.png"
    plt.figure(figsize=(10, 4))
    for t in data.columns:
        plt.plot(data[t], label=t, linewidth=2)
    plt.title(f"Cours des actions - Secteur {sector}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(graph1_path)
    plt.close()

    graph2_path = f"{tmpdir}/base100.png"
    plt.figure(figsize=(10, 4))
    for t in base100.columns:
        plt.plot(base100[t], label=t, linewidth=2)
    plt.title(f"Performance (Base 100) - Secteur {sector}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(graph2_path)
    plt.close()

    corr_path = f"{tmpdir}/corr.png"
    plt.figure(figsize=(6, 5))
    sns.heatmap(corr_matrix_named, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Matrice de corrélation")
    plt.tight_layout()
    plt.savefig(corr_path)
    plt.close()

    pdf = PDF()
    pdf.set_auto_page_break(auto=True, margin=15)

    # --- Page d intro
    pdf.add_page()
    pdf.set_fill_color(40, 65, 110)
    pdf.rect(0, 0, 210, 297, 'F')
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Arial", 'B', 28)
    pdf.ln(90)
    pdf.cell(0, 20, safe_text("Analyse Financière"), ln=True, align='C')
    pdf.set_font("Arial", 'I', 18)
    pdf.cell(0, 10, safe_text(f"Secteur : {sector}"), ln=True, align='C')
    pdf.set_font("Arial", '', 12)
    pdf.ln(100)
    pdf.set_y(-30)
    pdf.set_x(-60)
    pdf.set_text_color(220, 220, 220)
    pdf.set_font("Arial", 'I', 10)
    pdf.cell(0, 10, "Made by Youssef Chlih", 0, 0, 'R')
    pdf.set_text_color(0, 0, 0)

    # --- Page des Graphiques
    pdf.add_page()
    pdf.set_font("Arial", 'B', 14)
    pdf.set_fill_color(230, 230, 250)
    pdf.cell(0, 10, "Graphiques du secteur", ln=True, fill=True)
    pdf.image(graph1_path, w=180)
    pdf.ln(5)
    pdf.image(graph2_path, w=180)

    # --- Page Matrice
    pdf.add_page()
    pdf.set_font("Arial", 'B', 14)
    pdf.set_fill_color(230, 230, 250)
    pdf.cell(0, 10, "Matrice de corrélation", ln=True, fill=True)
    pdf.image(corr_path, w=170)

    # --- Page d entrepr
    pdf.add_page()
    pdf.set_font("Arial", 'B', 14)
    pdf.set_fill_color(230, 230, 250)
    pdf.cell(0, 10, "Fiche Descriptive des Entreprises", ln=True, fill=True)
    pdf.set_font("Arial", size=10)
    pdf.ln(5)
    for _, row in fiche_df.iterrows():
        ticker = row['Ticker']
        try:
            logo_url = yf.Ticker(ticker).info.get('logo_url')
            if logo_url:
                response = requests.get(logo_url)
                if response.status_code == 200:
                    img = Image.open(BytesIO(response.content))
                    img_path = f"{tmpdir}/{ticker}_logo.png"
                    img.save(img_path)
                    pdf.image(img_path, w=15, h=15)
                    pdf.set_x(pdf.get_x() + 3)
        except Exception:
            pass

        for col in fiche_df.columns:
            pdf.set_font("Arial", 'B', 10)
            pdf.cell(55, 8, safe_text(f"{col} :"), ln=0)
            pdf.set_font("Arial", '', 10)
            pdf.cell(0, 8, safe_text(row[col]), ln=1)
        pdf.ln(4)

    # Page lekhra
    pdf.add_page()
    pdf.set_fill_color(245, 245, 245)
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, "Conclusion", ln=True, fill=True)
    pdf.set_font("Arial", '', 12)
    pdf.ln(5)
    conclusion = generer_conclusion_ia(sector, fiche_df, base100, corr_matrix_named, profil, montant, np.log(data / data.shift(1)).dropna())

    try:
        profit_line = conclusion.split("\n")[-1]
        reste = "\n".join(conclusion.split("\n")[:-1])
        pdf.multi_cell(0, 10, safe_text(reste))
        pdf.ln(5)
        pdf.set_font("Arial", 'B', 12)
        pdf.set_fill_color(230, 255, 230)
        pdf.set_text_color(0, 100, 0)
        pdf.multi_cell(0, 10, safe_text(profit_line), border=1, fill=True, align='C')
        pdf.set_text_color(0, 0, 0)

    except Exception:
        pdf.multi_cell(0, 10, safe_text(conclusion))

    pdf.output(f"Analyse_financiere_{sector}.pdf")
# --- Bouton
sector_button = widgets.Button(description="Analyser", button_style='success')

def on_sector_button_clicked(b):
    sector = secteur_dropdown.value
    profil = profil_dropdown.value
    montant = montant_input.value

    if not sector or not profil:
        print("Veuillez sélectionner un secteur et un profil d'investisseur.")
        return

    data, base100, corr_matrix_named, fiche_df = calculs_financiers(sector)
    export_pdf(sector, data, base100, corr_matrix_named, fiche_df, profil, montant)
    plot_sector_charts(sector)

    from google.colab import files
    files.download(f"Analyse financière en {sector}.xlsx")
    files.download(f"Analyse_financiere_{sector}.pdf")

sector_button.on_click(on_sector_button_clicked)
display(secteur_dropdown, profil_dropdown, montant_input, sector_button)